In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from keras.preprocessing.text import Tokenizer
from keras_preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from keras.models import Model
from keras.layers import Embedding, LSTM, Dense, Input, TimeDistributed, Activation, dot, concatenate
from keras.callbacks import ModelCheckpoint
import spacy
 
# Read text file and split into lines and columns
def read_text(filename):
    with open(filename, mode='rt', encoding='utf-8') as file:
        text = file.read()
    return text
 
def to_lines(text, max_lines=None):
    sents = text.strip().split('\n')
    sents = [i.split('\t')[:2] for i in sents]  # Only take the first two columns (English, German)
    if max_lines:
        sents = sents[:max_lines]  # Only take up to max_lines rows
    return sents
# Read text file and split into lines and columns
def read_text(filename):
    with open(filename, mode='rt', encoding='utf-8') as file:
        text = file.read()
    return text
 
def to_lines(text, max_lines=None):
    sents = text.strip().split('\n')
    sents = [i.split('\t')[:2] for i in sents]  # Only take the first two columns (English, German)
    if max_lines:
        sents = sents[:max_lines]  # Only take up to max_lines rows
    return sents
 
# Read and process data
data = read_text("Rawdata.txt")
deu_eng = to_lines(data, max_lines=10000)  # Use only the first 10,000 rows
deu_eng = np.array(deu_eng)
print(deu_eng[:10])
 
# Lowercase the sentences
for i in range(len(deu_eng)):
    deu_eng[i, 0] = deu_eng[i, 0].lower()
    deu_eng[i, 1] = deu_eng[i, 1].lower()
 
# Tokenization function
def tokenization(lines):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(lines)
    return tokenizer
 
# Tokenize the sentences
eng_tokenizer = tokenization(deu_eng[:, 0])
eng_vocab_size = len(eng_tokenizer.word_index) + 1
eng_length = max(len(line.split()) for line in deu_eng[:, 0])
print(f'English Vocabulary Size: {eng_vocab_size}, Max Length: {eng_length}')
 
deu_tokenizer = tokenization(deu_eng[:, 1])
deu_vocab_size = len(deu_tokenizer.word_index) + 1
deu_length = max(len(line.split()) for line in deu_eng[:, 1])
print(f'Deutsch Vocabulary Size: {deu_vocab_size}, Max Length: {deu_length}')
 
# Encode sequences
def encode_sequences(tokenizer, length, lines):
    seq = tokenizer.texts_to_sequences(lines)
    seq = pad_sequences(seq, maxlen=length, padding='post')
    return seq
 
# Split data into train and test sets
train, test = train_test_split(deu_eng, test_size=0.2, random_state=12)
trainX = encode_sequences(deu_tokenizer, deu_length, train[:, 1])
trainY = encode_sequences(eng_tokenizer, eng_length, train[:, 0])
testX = encode_sequences(deu_tokenizer, deu_length, test[:, 1])
testY = encode_sequences(eng_tokenizer, eng_length, test[:, 0])

[['Go.' 'Geh.']
 ['Hi.' 'Hallo!']
 ['Hi.' 'Grüß Gott!']
 ['Run!' 'Lauf!']
 ['Run.' 'Lauf!']
 ['Wow!' 'Potzdonner!']
 ['Wow!' 'Donnerwetter!']
 ['Duck!' 'Kopf runter!']
 ['Fire!' 'Feuer!']
 ['Help!' 'Hilfe!']]
English Vocabulary Size: 2196, Max Length: 5
Deutsch Vocabulary Size: 3648, Max Length: 8


In [168]:
def build_model(in_vocab, out_vocab, in_timesteps, out_timesteps, units):
    model = Sequential()
    model.add(Embedding(in_vocab, units, input_length=in_timesteps, mask_zero=True))
    model.add(LSTM(units))
    model.add(RepeatVector(out_timesteps))
    model.add(LSTM(units, return_sequences=True))
    model.add(Dense(out_vocab, activation='softmax'))
    return model
 
model = build_model(deu_vocab_size, eng_vocab_size, deu_length, eng_length, 512)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
 
# Save the model with the best validation loss
filename = '455.keras'
checkpoint = ModelCheckpoint(filename, monitor='val_loss', verbose=1, save_best_only=True, mode='min')
 
# Train the model
history = model.fit(trainX, trainY.reshape(trainY.shape[0], trainY.shape[1], 1),
                    epochs=100, batch_size=512,
                    validation_split=0.2,
                    callbacks=[checkpoint], verbose=1)

Epoch 1/100
13/13 [==============================] - ETA: 0s - loss: 5.0493 - accuracy: 0.4082
Epoch 00001: val_loss improved from inf to 3.82521, saving model to 455.keras
13/13 [==============================] - 12s 757ms/step - loss: 5.0493 - accuracy: 0.4082 - val_loss: 3.8252 - val_accuracy: 0.4850
Epoch 2/100
13/13 [==============================] - ETA: 0s - loss: 3.6215 - accuracy: 0.4808
Epoch 00002: val_loss improved from 3.82521 to 3.43533, saving model to 455.keras
13/13 [==============================] - 13s 993ms/step - loss: 3.6215 - accuracy: 0.4808 - val_loss: 3.4353 - val_accuracy: 0.4848
Epoch 3/100
13/13 [==============================] - ETA: 0s - loss: 3.3194 - accuracy: 0.4824
Epoch 00003: val_loss improved from 3.43533 to 3.27486, saving model to 455.keras
13/13 [==============================] - 12s 947ms/step - loss: 3.3194 - accuracy: 0.4824 - val_loss: 3.2749 - val_accuracy: 0.4915
Epoch 4/100
13/13 [==============================] - ETA: 0s - loss: 3.1932 -

In [17]:
'''# Plot training history
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()
 '''
# Load SpaCy models for English and German
nlp_eng = spacy.load('en_core_web_sm')
nlp_ger = spacy.load('de_core_news_sm')
 
# Function for POS tagging and NER using SpaCy
def pos_ner_spacy(sentences, nlp):
    pos_tags = []
    entities = []
    for sent in sentences:
        doc = nlp(str(sent))  # Ensure the input is a native Python string
        pos_tags.append([(token.text, token.pos_) for token in doc])
        entities.append([(ent.text, ent.label_) for ent in doc.ents])
    return pos_tags, entities
 
# Get POS tags and NER for English sentences
eng_sentences = deu_eng[:, 0]
eng_pos_tags, eng_entities = pos_ner_spacy(eng_sentences, nlp_eng)
 
# Get POS tags and NER for German sentences
ger_sentences = deu_eng[:, 1]
ger_pos_tags, ger_entities = pos_ner_spacy(ger_sentences, nlp_ger)
 
# Display some examples
print("English POS Tags and Entities:")
for i in range(3):
    print(f"Sentence: {eng_sentences[i]}")
    print(f"POS Tags: {eng_pos_tags[i]}")
    print(f"Entities: {eng_entities[i]}")
    print()
 
print("German POS Tags and Entities:")
for i in range(3):
    print(f"Sentence: {ger_sentences[i]}")
    print(f"POS Tags: {ger_pos_tags[i]}")
    print(f"Entities: {ger_entities[i]}")
    print()


from keras.models import load_model
import numpy as np


 
# Create reverse lookup dictionaries
eng_index_word = {index: word for word, index in eng_tokenizer.word_index.items()}
deu_index_word = {index: word for word, index in deu_tokenizer.word_index.items()}
 
# Predict translation for a sample sentence
sample_source_seq = encode_sequences(deu_tokenizer, deu_length, [deu_eng[0, 1]])
predicted_sentence = predict_sequence(model, eng_tokenizer, sample_source_seq, eng_index_word, eng_length)
print(f"Source (German): {deu_eng[0, 1]}")
print(f"Predicted (English): {predicted_sentence}")
print(f"Actual (English): {deu_eng[0, 0]}")


English POS Tags and Entities:
Sentence: go.
POS Tags: [('go', 'VERB'), ('.', 'PUNCT')]
Entities: []

Sentence: hi.
POS Tags: [('hi', 'PROPN'), ('.', 'PUNCT')]
Entities: []

Sentence: hi.
POS Tags: [('hi', 'PROPN'), ('.', 'PUNCT')]
Entities: []

German POS Tags and Entities:
Sentence: geh.
POS Tags: [('geh', 'PROPN'), ('.', 'PUNCT')]
Entities: []

Sentence: hallo!
POS Tags: [('hallo', 'PROPN'), ('!', 'PUNCT')]
Entities: [('hallo!', 'ORG')]

Sentence: grüß gott!
POS Tags: [('grüß', 'ADV'), ('gott', 'NOUN'), ('!', 'PUNCT')]
Entities: [('grüß', 'LOC')]



KeyError: '<start>'

In [16]:
from keras.models import load_model

def predict_sequence(model, tokenizer, source_seq, index_word, max_length):
    predicted_sequence = []
    
    # Initialize the target sequence with '<start>'
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = tokenizer.word_index['<start>']
    
    for i in range(max_length):
        # Predict the next word
        output_token = model.predict(source_seq, verbose=0)
        
        # Get the index of the predicted word
        predicted_index = np.argmax(output_token[0, i, :])
        
        # Get the predicted word (handling KeyError)
        predicted_word = index_word.get(predicted_index, '<unknown>')
        
        # If the predicted word is '<end>', stop the loop
        if predicted_word == '<end>':
            break
        
        # Append the predicted word to the sequence
        predicted_sequence.append(predicted_word)
        
        # Update the target sequence for the next iteration
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = predicted_index
    
    return ' '.join(predicted_sequence)
# Load the trained model
model = load_model('455.keras')

# Define the input sequence (e.g., a German sentence)
input_sequence = "Versteck dich!"

# Preprocess the input sequence (e.g., tokenize and pad)
source_seq = encode_sequences(deu_tokenizer, deu_length, [input_sequence])

# Generate the predicted sequence
predicted_sentence = predict_sequence(model, eng_tokenizer, source_seq, eng_index_word, eng_length)
print(f"Input (German): {input_sequence}")
print(f"Predicted (English): {predicted_sentence}")

KeyError: '<start>'